In [ ]:
pip install llama-index llama-index-llms-openai python-dotenv tiktoken colorama nltk

Install Langsmith SDK

In [ ]:
pip install -U openai langsmith

## Setup

**Loading Environment Variables**

In [ ]:
from getpass import getpass
from dotenv import load_dotenv

import os

load_dotenv(override=True)  # Load environment variables from .env file

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY", "")  # Set OPENAI_API_KEY from environment variable
if not os.environ["OPENAI_API_KEY"]:
    os.environ["OPENAI_API_KEY"] = getpass("Enter OpenAI API key:")
    

**OpenAI Secret Key**

In [ ]:
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.llms.openai import OpenAI

embed_model = OpenAIEmbedding(
    api_key=os.getenv("OPENAI_API_KEY")
)
# -----------------------------
# LLM Setup
# -----------------------------
llm = OpenAI(model="gpt-4o-mini", temperature=0)

**Tracing System**

In [ ]:

# -----------------------------
# TRACING SYSTEM (LangSmith-style)
# -----------------------------
import uuid

class Tracer:
    def __init__(self):
        self.trace_id = str(uuid.uuid4())

    def log(self, step, data):
        print(f"\n🔍 TRACE [{self.trace_id}] - {step}")
        print(data)

tracer = Tracer()

In [ ]:
# -----------------------------
# DATA INGESTION
# ----------------------------

!mkdir -p 'data/paul_graham/'
!wget 'https://raw.githubusercontent.com/run-llama/llama_index/main/docs/examples/data/paul_graham/paul_graham_essay.txt' -O 'data/paul_graham/paul_graham_essay.txt'
!wget 'https://raw.githubusercontent.com/run-llama/llama_index/refs/heads/main/docs/examples/data/paul_graham/paul_graham_essay_ambiguated.txt' -O 'data/paul_graham/paul_graham_essay-ambiguated.txt'

In [ ]:

from llama_index.core import SimpleDirectoryReader
from llama_index.core.node_parser import SentenceSplitter
from langsmith import traceable

documents = SimpleDirectoryReader("./data/paul_graham/").load_data()

splitter = SentenceSplitter(
    chunk_size=512,
    chunk_overlap=50
)
nodes = splitter.get_nodes_from_documents(documents)

**Build Index**

In [ ]:
from llama_index.core import VectorStoreIndex
from llama_index.embeddings.openai import OpenAIEmbedding
import nltk

embed_model = OpenAIEmbedding()

index = VectorStoreIndex.from_documents(
    documents,
    transformations=[splitter],
    embed_model=embed_model
)

**Create a docstore**

[Documentation](https://developers.llamaindex.ai/python/framework/integrations/retrievers/bm25_retriever/#bm25-retriever--docstore-persistence)
- initialize a docstore to store nodes
- also available are mongodb, redis, postgres, etc for docstores

In [ ]:
# Docstore
from llama_index.core.storage.docstore import SimpleDocumentStore

docstore = SimpleDocumentStore()
docstore.add_documents(nodes)

## Hybrid Search (Vector + Keyword)

In [ ]:
from llama_index.core.retrievers import BaseRetriever
import Stemmer
from llama_index.retrievers.bm25 import BM25Retriever

vector_retriever = index.as_retriever(similarity_top_k=5)

# Simple keyword fallback (naive example)
keyword_retriever = index.as_retriever(similarity_top_k=5)

bm25_retriever = BM25Retriever.from_defaults(
    docstore=docstore,
    similarity_top_k=2,
    # Optional: We can pass in the stemmer and set the language for stopwords
    # This is important for removing stopwords and stemming the query + text
    # The default is english for both
    stemmer=Stemmer.Stemmer("english"),
    language="english",
)

# Re-define to implement the required abstract method `_retrieve`
class HybridRetriever(BaseRetriever):
    def __init__(self, vector_retriever, keyword_retriever):
        super().__init__()
        self.vector_retriever = vector_retriever
        self.keyword_retriever = keyword_retriever

    def _retrieve(self, query_bundle):
        query_str = (
            query_bundle.query_str
            if hasattr(query_bundle, "query_str")
            else str(query_bundle)
        )

        vector_nodes = self.vector_retriever.retrieve(query_str)
        keyword_nodes = self.keyword_retriever.retrieve(query_str)

        # Merge + deduplicate by node_id
        all_nodes = {n.node.node_id: n for n in vector_nodes}
        for n in keyword_nodes:
            all_nodes[n.node.node_id] = n

        return list(all_nodes.values())

hybrid_retriever = HybridRetriever(vector_retriever, keyword_retriever)

nodes = hybrid_retriever.retrieve("startup funding strategy")

In [ ]:
# -----------------------------
# MULTI-STAGE RETRIEVAL 
# -----------------------------

retriever = HybridRetriever(vector_retriever, keyword_retriever)

@traceable()
def multi_stage_retrieval_pipeline(query):
    """Example multi-stage retrieval pipeline with BM25 filtering and summarization."""
    return retriever.retrieve(query)

In [ ]:
# -----------------------------
# QUERY TRANSFORMATION
# -----------------------------

@traceable()
def transform_query(query):
    improved = f"Provide a precise and factual answer: {query}"
    prompt = f"""
    Rewrite the query to be more specific and suitable for document retrieval.
    
    Query: {query}
    
    Rewritten Query:
    """
    improved = llm.complete(prompt).text.strip()

    tracer.log("Query Transformation", improved)
    return improved

In [ ]:
# -----------------------------
# RERANK
# -----------------------------

from llama_index.core.postprocessor import SimilarityPostprocessor
from openai import OpenAI
import json

client = OpenAI()
reranker_llm = "gpt-5-reranker"

@traceable()
def re_rank_by_similarity(nodes, cutoff=0.6):
    """Re-rank nodes by similarity and filter out those below the cutoff."""
    postprocessor = SimilarityPostprocessor(similarity_cutoff=cutoff)
    return postprocessor.postprocess_nodes(nodes)

@traceable()
def llm_rerank(query, documents):
    """Reranks documents based on relevance to the query using an LLM."""
    ranked_results = []

    for i, doc in enumerate(documents):
        prompt = f"""
            You are a ranking system.

            Query: {query}
            Document: {doc}

            Return a JSON object with:
            - score (0-10)
            - reasoning (short)

            Only return valid JSON.
        """

        response = client.chat.completions.create(
            model=reranker_llm,
            messages=[{"role": "user", "content": prompt}],
        )

        content = response.choices[0].message.content

        try:
            parsed = json.loads(content)
            score = float(parsed["score"])
            reasoning = parsed.get("reasoning", "")
        except Exception:
            score = 0.0
            
            reasoning = "Parsing failed"

        # ✅ LOGGING
        print(f"\n--- Document {i} ---")
        print(f"Score: {score}")
        print(f"Reason: {reasoning[:200]}")
        print(f"Preview: {doc[:200]}")

        ranked_results.append((score, doc))

    return sorted(ranked_results, key=lambda x: x[0], reverse=True)


def advanced_llm_rerank(query, documents):
    results = []

    for i, doc in enumerate(documents):
        prompt = f"""
            You are an expert reranker.

            Evaluate this document against the query using:

            - relevance (0-5)
            - completeness (0-3)
            - accuracy (0-2)

            Query: {query}
            Document: {doc}

            Return ONLY JSON:
            {{
            "relevance": int,
            "completeness": int,
            "accuracy": int,
            "total": int,
            "reasoning": "short explanation"
            }}
        """

        response = client.chat.completions.create(
            model=reranker_llm,
            messages=[{"role": "user", "content": prompt}],
        )

        content = response.choices[0].message.content

        try:
            parsed = json.loads(content)
            total = float(parsed["total"])
        except Exception:
            total = 0.0
            parsed = {"reasoning": "Parsing failed"}

        # ✅ LOGGING
        print(f"\n=== Document {i} ===")
        print(f"Scores: {parsed}")
        print(f"Preview: {doc[:200]}")

        results.append((total, doc))

    return sorted(results, key=lambda x: x[0], reverse=True)

In [ ]:
# -----------------------------
# SUMMARIZATION + TRUNCATION
# -----------------------------

import tiktoken

def count_tokens(text):
    enc = tiktoken.encoding_for_model("gpt-4o-mini")
    return len(enc.encode(text))

@traceable()
def truncate_by_tokens(nodes, max_tokens=1500):
    """ Relevance-based Truncation: truncate the list of nodes to fit within the max token limit."""
    selected = []
    total_tokens = 0

    for node in nodes:
        node_text = node.text
        tokens = count_tokens(node_text)

        if total_tokens + tokens > max_tokens:
            break

        selected.append(node)
        total_tokens += tokens

    return selected

@traceable()
def summarize_context(filtered_nodes):
    """Context Optimization: summarize and compress  the context if it exceeds token limits."""
    total_tokens = sum(count_tokens(n.text) for n in filtered_nodes )

    if total_tokens > 1000:
        print("🧠 Applying summarization...")
        context = summarize_context(filtered_nodes)
    else:
        context = "\n\n".join([n.text for n in filtered_nodes])
    return context

In [ ]:
# -----------------------------
# LLM GENERATION
# -----------------------------

@traceable()
def generate_answer(query, context):
    """Generate an answer using the LLM based on the retrieved context."""
    prompt = f"""
        Answer the question using ONLY the context below.

        Context:
        {context}

        Question: {query}
        Answer:
    """
    return llm.complete(prompt)

In [ ]:
from colorama import Fore, Style

@traceable()
def multi_stage_query(query):
    """End-to-end optimized RAG pipeline."""
    
    tracer.log("User Query", query)
    
    # Step 1: Transform query
    transformed_query = transform_query(query)
    print(f"{Fore.CYAN}Stage 1 (✍️ Query Transformation){Style.RESET_ALL}: \n - original: {query}\n - transformed: {transformed_query}")
       
    # Step 2 — Multi-stage retrieval
    nodes = multi_stage_retrieval_pipeline(query)
    print(f"{Fore.CYAN}Stage 2 (🚀 Multi-stage retrieval){Style.RESET_ALL}: \n📄 Retrieved {len(nodes)} chunks")
    
    # Step 3 — Re-rank
    reranked_nodes = re_rank_by_similarity(nodes, cutoff=0.6)
    print(f"{Fore.CYAN}Stage 3 (📈 Re-ranking){Style.RESET_ALL}: {len(reranked_nodes)} nodes after filtering\n")

    # Step 4 — Context optimization
    filtered_nodes = truncate_by_tokens(reranked_nodes, max_tokens=1200)
    print(f"{Fore.CYAN}Stage 4 (✂️ Context Optimization){Style.RESET_ALL}: {len(filtered_nodes)} chunks nodes after truncation\n")
    context = summarize_context(filtered_nodes)
   
    # Step 5 — Final : Context Assembly + LLM Generation
    answer = generate_answer(query, context)
    print(f"{Fore.CYAN}Final (🤖 LLM Generation){Style.RESET_ALL}: \n{answer}")

    return answer

In [ ]:
# -----------------------------
# RUN AND STREAM PIPELINE
# -----------------------------

if __name__ == "__main__":
    print("\n🚀 Running Query...\n")
    multi_stage_query("Who is Paul Graham and what did he build?")